# 1-1 医用画像の種類とX線・CTの原理

範囲1 医用画像の種類と原理｜第2回

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nakaura-T/Medical_Imaging_Seminar_Public/blob/main/hiroshima/1_modalities/1-1_xray_ct.ipynb)

## この回で分かるようになること

- X線、CT、MRI、超音波、核医学の画像が、それぞれ何を画像にしているかを説明できる
- X線管でX線が発生するしくみ（制動X線と特性X線）と、管電圧・管電流の役割を説明できる
- X線の減弱と、物質の種類・厚さの関係を説明できる
- 光電効果とコンプトン散乱の違いと、骨やヨード造影剤が白く写る理由を説明できる
- CT装置の構成（X線管、検出器、ガントリー）を説明できる
- CTが投影データから断面画像を再構成する仕組みを、シミュレーションを使って説明できる
- 病気によってCT値が上がる理由と下がる理由を、密度と原子番号から説明できる
- ヨード造影剤でCT値が上がる理由と、低管電圧CT・デュアルエナジーCTのしくみを説明できる

## 使うデータ

- scikit-image付属の数値ファントム（Shepp-Logan ファントム。頭部の断面を楕円の組み合わせで模した画像）

## 0. 準備

**Colab で開いた場合**: 上の「Open In Colab」ボタンから開き、このまま次のセルを上から順に実行します。追加のインストールは不要です。

**VS Code で開いた場合**: ノートブック右上の「カーネルの選択」で `.venv` を選んでから、次のセルを実行します。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.data import shepp_logan_phantom
from skimage.transform import radon, iradon

## 1. 解説

### 1-1. 医用画像の種類と、それぞれが画像にしているもの

医用画像は、どれも「体の中の何か」を明るさに置き換えた画像です。何を置き換えているかは、撮影法（モダリティ）ごとに違います。

| モダリティ | 明るさに置き換えているもの | 得意な対象 |
|---|---|---|
| X線写真 | X線の通り抜けにくさを、X線が通った経路に沿って足し合わせたもの | 骨、肺野、腸管のガス |
| CT | 体の各点でのX線の通り抜けにくさ（CT値） | 臓器の形、出血、骨、肺 |
| MRI | 水素原子核（主に水と脂肪）が出す信号の強さ | 脳、脊髄、関節、軟部組織 |
| 超音波 | 組織の境界で跳ね返ってきた音波の強さ | 腹部臓器、心臓、胎児 |
| 核医学（PET、SPECT） | 体内に投与した放射性薬剤の分布 | 糖代謝や血流などの機能 |

X線写真とCTは、同じ「X線の通り抜けにくさ」を画像にしています。違いは、X線写真が経路上の値を足し合わせた結果しか持たないのに対し、CTは体の各点の値を持つことです。X線写真で心臓と背骨が重なって見えるのはこのためです。

では、CTは足し合わされた値から、どうやって各点の値を取り出しているのでしょうか。この回の後半で、シミュレーションを使って確かめます。

#### 胸部X線写真の例

![Chest X-ray PA 3-8-2010](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/xray_chest_pa_normal.jpg)

正面から撮った正常な胸部のX線写真です。骨はX線を受けやすく、経路上の足し合わせた値が大きいので白く、肺（主に空気）は通り抜けやすく黒く写ります。心臓や血管などの軟部組織は中間のグレーです。1-1の表で「経路上の値を足し合わせたもの」と書いたX線写真が、実画像でどのようになるかの例です。

出典: Wikimedia Commons — [Chest Xray PA 3-8-2010.png](https://commons.wikimedia.org/wiki/File:Chest_Xray_PA_3-8-2010.png) ／ 作者: Stillwaterising ／ ライセンス: [CC0](https://creativecommons.org/publicdomain/zero/1.0/)

### 1-2. X線の発生：X線管、管電圧と管電流

X線は、波長が短く、エネルギーの大きい電磁波です。X線の光子1個のエネルギーはkeV（キロ電子ボルト）で表します。1 eVは、電子1個を1 Vの電圧で加速したときに得るエネルギーです。診断に使うX線はおよそ20〜150 keVで、目に見える光（2〜3 eV）の数万倍のエネルギーをもちます。X線写真もCTも、このX線をX線管という真空管で作ります。

#### X線管の構造

![Structure of an X-ray tube](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/figures/fig_xray_tube.png)

X線管は、真空の容器の中に陰極（−）と陽極（+）を向かい合わせて置いたものです。容器の中を真空にするのは、電子が気体の分子にぶつからずに陽極まで届くようにするためです。

1. **陰極（フィラメント）**：タングステンの細い線を巻いたもので、電流を流して2000℃以上に熱すると、表面から電子が飛び出します（熱電子放出）。フィラメントの温度で、飛び出す電子の数が決まります。周りの集束カップが、電子を陽極の小さな一点に集めます
2. **管電圧**：陰極と陽極のあいだに数十〜百数十kVの高い電圧をかけ、電子を加速します。120 kVで加速された電子は、陽極に着くまでに120 keVの運動エネルギーをもちます
3. **陽極（ターゲット）**：加速された電子がタングステンの板に当たり、X線が出ます。電子が当たる点を焦点と呼び、焦点が小さいほど鋭い画像になります。ただし、電子のエネルギーのうちX線になるのは1%程度で、残りの約99%は熱になります。そこで陽極を円板にして毎分数千回転させ、電子が当たる場所を円周上に分散させて焦点が溶けないようにしています（回転陽極）
4. **窓・フィルタ・コリメータ**：X線は窓から外に出て、アルミニウムや銅のフィルタを通ります。フィルタは、体に吸収されるだけで検出器まで届かない低いエネルギーのX線を先に取り除き、無駄な被ばくを減らします。コリメータは、X線を撮影に必要な範囲だけに絞ります

#### タングステンを使う理由

陽極にもフィラメントにもタングステン（W）を使うのには、次の理由があります。

- **原子番号が大きい**（$Z = 74$）：次に説明する制動X線の発生効率は、原子番号と管電圧に比例します。原子番号の大きい金属ほど、同じ電子から多くのX線が出ます
- **融点が高い**（約3400℃。金属の中で最も高い）：焦点には熱が集中するので、溶けにくい金属が必要です。フィラメントも高温に熱して使うので、同じ性質が役に立ちます
- **熱を伝えやすく、高温でも蒸発しにくい**：焦点の熱を速く逃がせ、蒸発した金属で管の内側が汚れにくくなります

なお、乳房のX線撮影（マンモグラフィ）では、モリブデン（Mo）やロジウム（Rh）の陽極を使う装置があります。次に説明する特性X線のエネルギー（Moでは約17〜20 keV）が、乳房の軟部組織の差を写すのに適しているからです。

出典: 自作（matplotlib、生成スクリプト `instructor/make_figures_1_1.py`）／ 作者: 本教材の作成者／ ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0)

#### 制動X線と特性X線

陽極に当たった電子からX線が出るしくみは、2つあります。

**制動X線**（制動放射、bremsstrahlung）は、電子がタングステンの原子核の近くを通るときに生じます。原子核のプラスの電荷に引かれて電子の進路が曲がり、速度が落ちます。このとき失った運動エネルギーが、X線の光子として放出されます。進路のわずかな曲がりから、ほぼ止まるまで、失うエネルギーの大きさはさまざまなので、制動X線のエネルギーは0から最大値まで連続的に分布します（連続スペクトル）。最大値は、電子が全エネルギーを一度に失った場合で、管電圧の値に一致します（120 kVなら120 keV）。X線管から出るX線の大部分は、この制動X線です。

**特性X線**は、電子がタングステンの原子の内側の軌道（K殻）にある電子を弾き飛ばしたときに生じます。空いた席に外側の軌道（L殻、M殻）の電子が落ち込み、2つの軌道のエネルギーの差がX線の光子として放出されます。軌道のエネルギーは元素ごとに決まっているので、特性X線のエネルギーも元素に特有の決まった値になります。「特性」という名前はこのためです。タングステンでは、L殻から落ちるKα線が58〜59 keV、M殻などから落ちるKβ線が67〜69 keVです。K殻の電子を弾き飛ばすには、K殻の結合エネルギー（タングステンで69.5 keV）以上のエネルギーが必要なので、Kの特性X線は管電圧が69.5 kVを超えたときにだけ出ます。

内殻の電子が弾き飛ばされて空いた席を外側の電子が埋めるしくみは、1-4で扱う光電効果のあとにも起こります。また、1-4のヨードのK吸収端（33.2 keV）は、ヨードのK殻の結合エネルギーにあたります。

![X-ray spectra for different tube voltages and currents](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/figures/fig_xray_spectrum.png)

タングステン陽極から出て、厚さ6 mmのアルミニウムのフィルタを通ったあとのX線のスペクトルを、簡単なモデル（制動X線はKramersの式）で計算した図です。特性X線のピークの高さは模式的な値です。

- (a) 管電流を同じにして、管電圧を80、120、140 kVに変えたもの。スペクトルの右端（▼）は、管電圧の値に一致します。20 keVより低いX線がほとんどないのは、フィルタが取り除いたためです。Kα線とKβ線のピークは、管電圧によらず同じエネルギーの位置に出ます
- (a) の凡例の平均エネルギーは、80 kVで46 keV、120 kVで58 keV、140 kVで63 keVです。1-3で線減弱係数の目安に使う「60 keV付近」は、CTでよく使う120 kVの平均エネルギーに近い値です。管電圧を上げると光子の数も大きく増え、このモデルでは80 kVの光子の数は120 kVの約3割です
- (b) 管電圧を同じにして、管電流を2倍にしたもの。スペクトルの形は変わらず、高さだけが2倍になります

出典: 自作（matplotlib、生成スクリプト `instructor/make_figures_1_1.py`。アルミニウムの減弱係数は NIST XCOM の値）／ 作者: 本教材の作成者／ ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0)

#### 管電圧と管電流の役割

撮影のときに設定する主な条件は、管電圧と管電流です。

| | 管電圧（kV） | 管電流（mA）と管電流時間積（mAs） |
|---|---|---|
| 決めるもの | 電子を加速する電圧。X線のエネルギー（最大値と平均） | 1秒間に陽極に当たる電子の数。X線の光子の数 |
| スペクトルの形 | 変わる（高いエネルギーの側へ広がる） | 変わらない（高さだけが変わる） |
| 画像への影響 | 上げると体を通り抜けやすくなるが、光電効果が減るので組織のコントラストは下がる（1-4） | 上げるとノイズが減る |
| 被ばく | 上げると増える | 管電流時間積に比例して増える |
| CTでの目安 | 80〜140 kV（標準的には120 kV） | 数十〜数百mA、1回転0.25〜1秒 |

管電流時間積（mAs）は、管電流（mA）に撮影時間（秒）を掛けた値で、その撮影で陽極に当たった電子の総数、つまり発生したX線の光子の総数に比例します。

管電流時間積を変えると画像のノイズがどう変わるかは、CT値を導入したあとの1-6で確かめます。

### 1-3. X線の減弱（ランベルト・ベールの法則）

X線は物質を通り抜けるあいだに、吸収や散乱によって弱まります（減弱）。強さ $I_0$ のX線が、線減弱係数 $\mu$ の物質を厚さ $x$ だけ通り抜けると、強さは次のようになります。

$$I = I_0 \, e^{-\mu x}$$

$\mu$ は物質ごとに決まる値で、骨は軟部組織より大きく、空気はほぼ0です。X線のエネルギーが60 keV付近（1-2で見た、120 kVで撮影したX線の平均エネルギーに近い値）のときの目安は、脂肪が約0.18 /cm、水が約0.21 /cm、緻密な骨が約0.57 /cmです。

体の中では、X線は種類の違う組織を順に通り抜けます。このとき両辺の対数をとると、次の関係が成り立ちます。

$$\ln \frac{I_0}{I} = \mu_1 x_1 + \mu_2 x_2 + \cdots$$

左辺は検出器で測れる量です。右辺は、経路上の $\mu$ に厚さを掛けて足し合わせたものです。X線写真の1つの画素が表しているのは、この「足し合わせた値」（投影値）です。

#### 減弱と投影値の模式図

![X-ray attenuation through layers and projection value](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/figures/fig_attenuation.png)

X線が脂肪・軟部組織・骨の層を順に通り抜ける様子を示した模式図です。矢印の太さはX線の強さを表し、層を通るたびに細くなります。検出器で測れる $\ln(I_0/I)$ は、経路上の（線減弱係数 $\mu$ × 厚さ $x$）を足し合わせた投影値です。

出典: 自作（matplotlib、生成スクリプト `instructor/make_figures_1_1.py`。`uv run python instructor/make_figures_1_1.py` で作り直せます）／ 作者: 本教材の作成者／ ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0)

### 1-4. X線が弱まる2つの仕組み：光電効果とコンプトン散乱

1-3で挙げた値には、まだ説明のついていない差があります。緻密な骨の $\mu$ は0.57 /cmで、水の0.21 /cmの約2.7倍です。ところが、骨の密度は水の約1.9倍しかありません。X線を弱めるのが物質の詰まり具合だけなら、骨は1.9倍で止まるはずです。

診断に使うエネルギー帯（およそ30〜120 keV）で、X線が弱まる原因は主に2つです。

**光電効果**は、X線の光子が原子の内側の電子にエネルギーをすべて渡して消える現象です。電子は原子から弾き出され、光子はその場でなくなります。起こりやすさは、物質の原子番号 $Z$ のおよそ3乗に比例し、光子のエネルギー $E$ のおよそ3乗に反比例します。$Z$ が大きい物質ほど、そしてエネルギーが低いほど、急に起こりやすくなります。

**コンプトン散乱**は、光子が外側の電子と衝突し、エネルギーの一部を渡して向きを変える現象です。光子は消えずに、弱くなって別の方向へ飛んでいきます。起こりやすさは経路にある電子の数、つまりほぼ物質の密度で決まり、原子番号にはあまりよりません。エネルギーが上がっても、光電効果ほど急には減りません。

$\mu$ は、この2つ（と、わずかな割合の干渉性散乱）の起こりやすさを足したものです。骨に戻ると、2.7倍の内訳はこうなります。水より詰まっている分（密度で1.9倍）はコンプトン散乱を増やし、骨に多いカルシウム（$Z = 20$。水を作る水素と酸素は1と8）は光電効果を増やします。60 keV付近では、残りの1.4倍がカルシウムによる光電効果の分です。エネルギーを下げるほど光電効果の割合は大きくなり、上げるほど小さくなります。

原子番号がほぼ同じ組織どうし、たとえば脂肪と筋肉、肝臓と脾臓では、光電効果に差がほとんどありません。残るのは電子の数、つまり密度の差だけで、CT値の違いも数十HUにとどまります。軟部組織の病変を見るのに造影剤が要るのは、このためです。

2つの割合が物質とエネルギーで変わることは、撮影条件の選び方と、画像に出るアーチファクトの両方に現れます。

- **造影剤と管電圧**：ヨード（$Z = 53$）は光電効果が大きく、33.2 keVをわずかに超えたところでK殻の電子を弾き飛ばせるようになり、光電効果がさらに急に大きくなります（K吸収端）。このエネルギーによる違いを使う低管電圧CTとデュアルエナジーCTは、1-8で扱います
- **散乱線を減らす**：コンプトン散乱で向きを変えた光子が検出器に入ると、その光子が通っていない経路の情報として数えられ、コントラストが下がります。X線写真のグリッドや、CTのコリメータと散乱線補正は、これを減らすためのものです
- **ビームハードニング**：1-3の式は $\mu$ を一定として扱っていましたが、X線管から出るのは連続スペクトルです（1-2）。低いエネルギーの成分ほど光電効果で先に減らされるので、物質を通り抜けるほどビームの平均エネルギーは上がり、残った光子に対する $\mu$ は小さくなります。$\ln(I_0/I)$ は厚さに正比例しなくなり、均一な物体でも中心のCT値が周辺より低く出ます（カッピング）。骨や金属のあいだに出る暗い帯も、同じ原因によるものです。2-4で、この沈み込みを再構成画像で確かめます

#### 光電効果とコンプトン散乱の模式図

![Photoelectric effect and Compton scattering](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/figures/fig_interactions.png)

左が光電効果（X線の光子が電子にエネルギーをすべて渡して、光子が消える現象）、右がコンプトン散乱（光子が電子と衝突し、エネルギーの一部を渡して向きを変える現象）です。光電効果の起こりやすさは原子番号の3乗に比例し、コンプトン散乱の起こりやすさは経路にある電子の数（ほぼ密度）で決まります。

出典: 自作（matplotlib、生成スクリプト `instructor/make_figures_1_1.py`。`uv run python instructor/make_figures_1_1.py` で作り直せます）／ 作者: 本教材の作成者／ ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0)

#### ビームハードニングの実例

![Beam hardening artifact caused by a hip prosthesis](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/ct_beam_hardening_metal.jpg)

人工股関節（大腿骨に埋め込んだ金属の関節）がある部位を撮影したCTの横断面です。白い人工関節の左右に伸びる暗い帯が、ビームハードニング（連続スペクトルのX線で、低エネルギーの成分が先に吸収されるために生じるアーチファクト）です。

出典: Wikimedia Commons — [Aufhaertungsartefakte in der CT durch HTEP 86W - CT axial - 001.jpg](https://commons.wikimedia.org/wiki/File:Aufhaertungsartefakte_in_der_CT_durch_HTEP_86W_-_CT_axial_-_001.jpg) ／ 作者: Hellerhoff ／ ライセンス: [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0)

### 1-5. CT装置と投影データ

#### ガントリーと検出器

CT装置は、患者さんが横たわる寝台と、寝台が通り抜けるドーナツ形のガントリーでできています。ガントリーの中では、X線管と検出器が向かい合わせに回転フレームに取り付けられ、体の周りを1回転0.25〜0.5秒程度で回ります。

![CT gantry and detector elements](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/figures/fig_ct_gantry.png)

- **X線管**：1-2で見た回転陽極のX線管です。CTでは大きな管電流を連続して流すので、陽極の熱容量が大きく、冷却能力の高いものを使います
- **ボウタイフィルタとコリメータ**：体は中心が厚く周辺が薄いので、周辺に向かうX線を弱めるフィルタ（形が蝶ネクタイに似ていることからボウタイと呼ぶ）で、周辺の余分な被ばくを減らし、検出器に届く強さをそろえます。コリメータは、体の長さ方向のX線の幅を絞ります。X線は扇形に広がって体を通り抜けます（ファンビーム）
- **検出器**：X線管の反対側に、多数の検出素子が円弧状に並んでいます。1列に700〜900個程度の素子（チャンネル）があり、その列が体の長さ方向に16〜320列並んだ多列検出器（MDCT）が一般的です。320列の装置では、1回転で体の長さ方向に約16 cmの範囲を撮影できます
- **スリップリング**：回転部分への電力の供給とデータの送信を、ケーブルではなく回転する接点で行います。そのため回転フレームは同じ向きに回り続けられ、寝台を一定の速さで動かしながら連続して撮影できます（ヘリカルスキャン。X線管が体の周りをらせん状に動くことからこう呼ぶ）

(b)は、検出素子の構造です。現在の多くのCTは間接変換型の検出器を使います。

1. 検出素子の手前に並んだ薄い板（散乱線除去用のコリメータ）が、斜めから入ってくる散乱線（1-4のコンプトン散乱で向きを変えた光子）を遮ります
2. シンチレータ（ガドリニウム酸硫化物のセラミックなど）が、X線の光子を受けて可視光を出します
3. シンチレータの下のフォトダイオードが、光を電流に変えます。電流は数値に変換されてコンピュータへ送られます

検出器が測っているのは、体を通り抜けたX線の強さ $I$ です。体がないときの強さ $I_0$ は、装置の校正（空気だけを撮影して測る）と、X線管の出力を監視する参照用の検出素子から求めます。装置はこの2つから $\ln(I_0/I)$ を計算し、1-3の投影値として再構成に使います。

最近は、X線の光子を半導体（テルル化カドミウムなど）で直接電気の信号に変え、光子を1個ずつ数えてそのエネルギーも測る光子計数CT（photon-counting CT）も、2021年ごろから臨床で使われ始めています。

出典（模式図）: 自作（matplotlib、生成スクリプト `instructor/make_figures_1_1.py`）／ 作者: 本教材の作成者／ ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0)

![Siemens SOMATOM go.Top CT scanner](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/photo_ct_scanner.jpg)

実際のCT装置の例（Siemens Healthineers SOMATOM go.Top）です。中央に穴の開いたドーナツ形の部分がガントリーで、上の模式図 (a) の回転フレーム、X線管、検出器はこのカバーの中に収められています。手前の寝台に患者さんが横たわり、寝台ごと穴の中を移動しながら撮影します。

出典: Wikimedia Commons — [Siemens Somatom CT scanner.jpg](https://commons.wikimedia.org/wiki/File:Siemens_Somatom_CT_scanner.jpg) ／ 撮影: USDA Rural Development ／ ライセンス: パブリックドメイン（Public domain）

#### CT装置の発展：アキシャルCTからヘリカルCT、多列検出器CTへ

最初の臨床用CTは、1971年にロンドンの病院で頭部の撮影に使われたEMIスキャナです。開発したHounsfieldは、再構成の数学的な基礎を築いたCormackとともに、1979年にノーベル生理学・医学賞を受けました。当初は細いX線を平行移動させながら少しずつ回転させる方式で、1枚の断面の撮影に数分かかりました。その後、扇形に広がるX線（ファンビーム）と円弧状の検出器を向かい合わせて一緒に回転させる方式（第3世代）が主流になり、現在のCTもこの方式です。

![Axial, helical and multi-slice helical CT, and the number of detector rows](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/figures/fig_ct_evolution.png)

- **(a) アキシャル撮影**（ステップ・アンド・シュート）：1990年ごろまでは、X線管と検出器にケーブルがつながっていたので、1回転したら止めて逆に巻き戻す必要がありました。1回転で1枚の断面を撮り、寝台を少し動かして止め、また1回転する、を繰り返します。1枚ごとに止まるので時間がかかり、息止めの間に広い範囲を撮れず、呼吸のずれで断面と断面のあいだに撮り残しが生じることもありました
- **(b) ヘリカル撮影**（らせん撮影）：上で見たスリップリングによって、回転部分を止めずに回し続けられるようになりました。回転させながら寝台を一定の速さで動かすと、X線管は体の周りをらせん状に動き、連続した体積のデータが得られます。1990年ごろから臨床で使われ始めました。1回転あたりの寝台の移動距離を、X線のビームの幅で割った値をピッチと呼び、ピッチが大きいほど速く広い範囲を撮影できます。らせん状のデータから、断面ごとのデータを補間して作ってから再構成します
- **(c) 多列検出器CT**（マルチスライスCT、MDCT）：体の長さ方向に検出器の列を並べ、1回転で複数の断面のデータを同時に集めます。1998年に4列の装置が登場し、16列、64列と増え、2007年ごろには320列（1回転で体の長さ方向に16 cm）の装置が登場しました（(d)）。1回転の時間も、初期のヘリカルCTの約1秒から0.25秒程度まで短くなりました

列数が増えて1回転が速くなったことで、薄い断面（0.5 mm程度）で広い範囲を数秒で撮影できるようになりました。縦・横・奥行きの画素の大きさがそろった（等方性の）体積データが得られるので、冠状断や矢状断、血管の立体表示を、横断面と同じ細かさで作れます。動いている心臓を撮る心臓CT（冠動脈CT）や、320列の装置で頭部全体を寝台を動かさずに繰り返し撮る灌流CTも、この発展によって可能になりました。さらに、2005年には2管球型のデュアルソースCT（1-8）、2021年ごろからは光子計数CTが臨床で使われ始めています。再構成の方法の発展は、1-3のノートブック（画質改善技術）で扱います。

出典: 自作（matplotlib による模式図。生成スクリプト `instructor/make_figures_1_1.py`）／ 作者: 本教材の作成者／ ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0)

![Philips Brilliance 64-slice CT seen from the control room](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/photo_ct_64slice.jpg)

64列の多列検出器CT（Philips Brilliance 64）を、ガラス越しの操作室から見た写真です。手前のモニターで撮影条件を設定し、再構成した画像を確認します。

出典: Wikimedia Commons — [64 Slice CT Scanner.jpg](https://commons.wikimedia.org/wiki/File:64_Slice_CT_Scanner.jpg) ／ 撮影: Glitzy queen00（英語版Wikipedia） ／ ライセンス: パブリックドメイン（Public domain）

![Siemens NAEOTOM Alpha photon-counting CT](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/photo_ct_photon_counting.jpg)

臨床で使われている光子計数CT（Siemens Healthineers NAEOTOM Alpha、チェコのプルゼニ大学病院）です。外観はふつうのCTと変わりませんが、ガントリーの中の検出器が、光子を1個ずつ数える半導体検出器になっています（1-8）。左は造影剤の自動注入器です。

出典: Wikimedia Commons — [CT Naeotom Alpha Pilsen 2022 (cropped).jpg](https://commons.wikimedia.org/wiki/File:CT_Naeotom_Alpha_Pilsen_2022_(cropped).jpg) ／ 撮影: Tomáš Vendiš ／ ライセンス: [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0)

#### 投影データとサイノグラム

CTでは、X線管と検出器が体の周りを回転し、さまざまな方向から投影値を集めます。

集めた投影値を、縦軸に検出器の位置、横軸に投影の角度をとって並べた画像を**サイノグラム**と呼びます。体の中の1点は、角度が変わると検出器上の位置が正弦波（サイン波）を描くように動くので、この名前が付いています。

#### 撮影幾何とサイノグラムの模式図

![CT acquisition, one projection, and sinogram](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/figures/fig_ct_geometry.png)

X線管と検出器が体の周りを回転させる撮影の幾何（左）、ある1つの角度（0度）での投影値（中央）、全角度の投影を並べたサイノグラム（右）を示します。サイノグラムは投影データを角度ごとに並べた画像で、体の中の1点は角度が変わると正弦波を描く軌跡（右図の明るい2本の曲線）になります。

出典: 自作（matplotlib、生成スクリプト `instructor/make_figures_1_1.py`。`uv run python instructor/make_figures_1_1.py` で作り直せます）／ 作者: 本教材の作成者／ ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0)

### 1-6. 再構成とCT値

#### フィルタ補正逆投影法

サイノグラムから断面画像を求める処理を**再構成**と呼びます。最も素朴な方法は、各方向の投影値を、X線が通った経路に沿って画像に塗り戻し、全方向分を足し合わせる方法です（逆投影）。

素朴な逆投影だけでは、画像がぼやけます。1つの点の値が、その点を通るすべての経路に塗り広げられるためです。

そこで、逆投影の前に投影値へフィルタ（輪郭を強めるフィルタ）をかけておくと、ぼやけが打ち消されて元の断面に近い画像が得られます。これを**フィルタ補正逆投影法**（FBP）と呼びます。現在のCT装置では、ノイズを減らせる逐次近似再構成も使われています。

#### 逆投影とフィルタ補正逆投影法の模式図

![Backprojection with increasing numbers of angles and filtered back projection](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/figures/fig_backprojection.png)

逆投影（各方向の投影値を、X線が通った経路に沿って画像に塗り戻し、全方向分を足し合わせる再構成法）を、角度の数を増やして適用した様子（左から1、3、16、180角度）と、投影値にフィルタをかけてから逆投影した結果（右端）を示します。フィルタなしでは角度を増やしても画面全体がぼやけたままになります。フィルタをかけると輪郭がはっきりし、元の断面に近づきます。

出典: 自作（matplotlib、生成スクリプト `instructor/make_figures_1_1.py`。`uv run python instructor/make_figures_1_1.py` で作り直せます）／ 作者: 本教材の作成者／ ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0)

#### CT値

再構成で得られた各点の $\mu$ を、水を基準に換算した値が**CT値**です。

$$\text{CT値} = 1000 \times \frac{\mu - \mu_{水}}{\mu_{水}} \quad \text{（単位 HU）}$$

この式から、水は0 HU、空気（$\mu \approx 0$）は −1000 HU になります。CT値は範囲2で詳しく扱います。

#### 正常な頭部CTの例

![CT of a normal brain, axial slice](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/ct_head_normal_axial.png)

正常な頭部を水平に切断したCT像（横断面）です。骨は白く、副鼻腔（鼻のまわりの空気が入った空間）の空気は黒く、脳は中間のグレーに写っています。これはCT値の高低が明るさに置き換わっている結果です。画像の右側は撮影位置の目安を示す矢状面（左右から身体を見た向き）の参照画像で、黄色い線が横断面の位置です。

出典: Wikimedia Commons — [CT of a normal brain, axial 20.png](https://commons.wikimedia.org/wiki/File:CT_of_a_normal_brain,_axial_20.png) ／ 作者: Mikael Häggström ／ ライセンス: [CC0](https://creativecommons.org/publicdomain/zero/1.0/)

#### 画像のノイズと管電流時間積

1-2で見たように、管電流時間積（mAs）は発生するX線の光子の数に比例します。検出器に届く光子の数は、同じ条件で測っても偶然によって毎回ばらつきます。このばらつきはポアソン分布に従い、平均 $N$ 個の光子を数えたときの標準偏差は $\sqrt{N}$ です。ばらつきの割合 $\sqrt{N}/N = 1/\sqrt{N}$ は、光子が多いほど小さくなります。このため、画像のノイズの標準偏差（SD）は $1/\sqrt{\mathrm{mAs}}$ に比例します。ノイズを半分にするには、mAsを4倍にする必要があり、被ばくも4倍になります。

![Image noise at 1x and 4x mAs](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/figures/fig_noise_mas.png)

直径20 cmの水の円柱に、骨・脂肪・軟部組織の棒を入れた模型を、光子の数のばらつきを含めて投影・再構成したシミュレーションです。橙の円の中のCT値のSD（ノイズ）は、1 × mAsの42 HUから、4 × mAsで20 HUとほぼ半分になっています。ノイズのない再構成（左）でも、再構成の計算の誤差で1 HUのばらつきがあります。

画質と被ばくはこのように引き換えの関係にあるので、CT装置は、体の厚さに合わせて管電流を自動で調整する機能（自動露出機構、AEC）をもっています。体の薄い部分や、X線が短い経路で通り抜ける向きでは管電流を下げ、厚い部分では上げることで、ノイズをそろえながら被ばくを抑えます。1-8で扱う低管電圧CTでは、光子が減ってノイズが増える分を、mAsを上げて補うことがあります。

出典: 自作（NumPy と scikit-image によるシミュレーション、matplotlib による作図。生成スクリプト `instructor/make_figures_1_1.py`）／ 作者: 本教材の作成者／ ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0)

### 1-7. 病気によるCT値の変化

1-6で見たように、CT値は画素ごとの $\mu$（線減弱係数）を、水を0とした目盛りで表したものです。組織の密度や成分が変われば $\mu$ も変わるので、病気はCT値を上下させます。同じ部位で向きが反対になる6つの例を、上昇と低下に分けた表にまとめます。値はすべて目安で、撮影条件や個人によって変わります。

| 部位（正常時の目安） | CT値が上がる例 | CT値が下がる例 |
|---|---|---|
| 肺（約−800 HU） | **肺炎**（肺胞に炎症で液体がたまる病気）<br>約−800 → 約+20 HU<br>肺胞の液体で密度と $\mu$ が増える | **肺気腫**（肺胞の壁が壊れて空気が過多にたまり、肺が膨らむ病気）<br>約−800 → 約−950 HU<br>空気の割合が増え、密度と $\mu$ が減る |
| 脳（38〜60 HU） | **脳出血**（脳の中で出血し、血液の塊＝血腫ができる病気）<br>約38 → 約65 HU<br>鉄を含む血液がたまって密度と $\mu$ が増える | **脳梗塞**（脳の血管が詰まって血流が止まり、組織がむくむ病気）<br>約38 → 約30 HU<br>むくみで水分が増え、密度と $\mu$ が減る |
| 肝臓（38〜60 HU） | **ヘモクロマトーシス**（鉄が体に過剰にたまる病気）<br>約60 → 約90 HU<br>鉄（原子番号26）が増え、光電効果（1-4）で $\mu$ が増える | **脂肪肝**（肝臓に脂肪がたまる病気）<br>約60 → 約25 HU<br>脂肪の $\mu$ は水より小さい（1-3）ため、値が下がる |

次の図は、表と同じ例をCT値の目盛りの上に並べたものです。

#### CT値スケールと病気で動く向き

![CT number (HU) scale and typical shifts with diseases](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/figures/fig_hu_scale.png)

CT値（HU。水を0とした、X線の通り抜けにくさの目盛り）の−1000〜+1000に、空気・肺・脂肪・水・軟部組織・骨を並べた目盛り（上）と、病気でCT値が動く向きの例（下）を示します。正常な肺（約−800 HU）は、肺炎で+20付近まで上がり、気腫（肺が大きく膨らむ病気）で−950付近まで下がります。肝臓や脳（正常38〜60 HU）では、出血（血腫）や鉄の過剰で上がり、梗塞や脂肪肝で下がります。

出典: 自作（matplotlib、生成スクリプト `instructor/make_figures_1_1.py`。`uv run python instructor/make_figures_1_1.py` で作り直せます）／ 作者: 本教材の作成者／ ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0)

#### 例：肺炎のCT（CT値が上がる）

![CT of lobar pneumonia](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/ct_pneumonia_lobar.jpg)

右中葉（右の肺をいくつかに分けた区域の1つ）に大葉性肺炎（肺の一葉全体に広がる肺炎）が起きたCTです。上の横断面では画像の左（CT画像は左右が反転しており、患者さんの右側）に濃い領域が見え、下は同じ部位を左右から見た冠状面と、前後から見た矢状面です。濃いところは実質化（肺胞が液体で埋まって軟部組織と同じように写る部分）で、空気が抜けて水に近い密度になるため $\mu$ が大きくなります。CT値は正常な肺（約−800 HU）から上へ動き、目安で約+20 HUです。

出典: Wikimedia Commons — [CT of lobar pneumonia.jpg](https://commons.wikimedia.org/wiki/File:CT_of_lobar_pneumonia.jpg) ／ 作者: Mikael Häggström ／ ライセンス: [CC0](https://creativecommons.org/publicdomain/zero/1.0/)

#### 例：脳出血のCT（CT値が上がる）

![CT of basal ganglionic hemorrhage](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/ct_brain_hemorrhage.png)

脳の深部で出血が起きたCTの横断面です。*印が血腫（血液の塊）で、白く（高CT値に）写り、矢印はその内側（中線側）の境界付近を示します。鉄を含む血液がたまることで密度と $\mu$ が増えるため、CT値は正常時の脳の例（約38 HU）から目安で約65 HUへ上がります。

出典: Wikimedia Commons — [CT of basal ganglionic hemorrhage.png](https://commons.wikimedia.org/wiki/File:CT_of_basal_ganglionic_hemorrhage.png) ／ 作者: Shazia Mirza and Sankalp Gokhale ／ ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)

#### 例：ヘモクロマトーシス（CT値が上がる）

ヘモクロマトーシスは、鉄が体に過剰にたまって肝臓などの臓器を傷める病気です（概要は[英語版Wikipediaの Iron overload](https://en.wikipedia.org/wiki/Iron_overload)で確認できます）。肝臓に鉄がたまると、造影剤を使わない撮影（非造影CT）で肝臓のCT値が上がることが知られています。1994年の報告では、患者さんの肝臓の平均が79 HU、対照群が61 HUでした（[PubMed: 8079280](https://pubmed.ncbi.nlm.nih.gov/8079280/)）。本教材の目安では、正常な肝臓（約60 HU）から約90 HUへ上がります。理由は、鉄（原子番号26）が増えると光電効果が強くなって $\mu$ が大きくなるためです（1-4）。

再利用できるライセンスの臨床CT画像をWikimedia Commonsで見つけられなかったため、臨床画像は掲載せず、代わりに同じ向きの変化を示す自作図を置きます。

![Liver CT number: normal, iron overload, and fatty liver](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/figures/fig_liver_hu_pairs.png)

肝臓のCT値の目安を、正常・鉄過剰・脂肪肝で比べた自作図です。鉄過剰（ヘモクロマトーシス）で上がり、脂肪肝で下がるという向きは、1-7の最初の図（CT値スケールと病気で動く向き）と同じです。

出典: 自作（matplotlib、生成スクリプト `instructor/make_figures_1_1.py`。`uv run python instructor/make_figures_1_1.py` で作り直せます）／ 作者: 本教材の作成者／ ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0)

#### 例：脂肪肝のCT（CT値が下がる）

![Normal liver and fatty liver on CT](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/ct_liver_normal_vs_steatosis.jpg)

同じ患者さんの肝臓を比べたCTです。上が正常な肝臓で、下は抗がん治療の過程でできた脂肪肝です。下の画像の肝臓は周囲よりやや暗く（低CT値に）写っています。1-3の目安のとおり脂肪の $\mu$（約0.18 /cm）は水（約0.21 /cm）より小さいので、肝臓に脂肪がたまるとCT値は下がります。正常な肝臓（約60 HU）から目安で約25 HUへ下がります。

出典: Wikimedia Commons — [Vergleich normale Leber - Steatosis - CT axial 001.jpg](https://commons.wikimedia.org/wiki/File:Vergleich_normale_Leber_-_Steatosis_-_CT_axial_001.jpg) ／ 作者: Hellerhoff ／ ライセンス: [CC BY-SA 3.0](https://creativecommons.org/licenses/by-sa/3.0/)

#### 例：脳梗塞のCT（CT値が下がる）

![CT of cerebral infarction](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/ct_brain_infarction.png)

脳梗塞（脳の血管が詰まって血流が止まり、その先の組織が障害される病気）のCT横断面です。矢印は、画像の右側（患者さんの左）で周囲よりやや暗くなっている領域の目印です。梗塞が起きた組織はむくんで水分が増えるため、密度と $\mu$ が下がります。CT値は正常時の脳の例（約38 HU）から目安で約30 HUへ下がります。

出典: Wikimedia Commons — [CT of cerebral infarction.png](https://commons.wikimedia.org/wiki/File:CT_of_cerebral_infarction.png) ／ 作者: Shazia Mirza and Sankalp Gokhale ／ ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)

#### 例：肺気腫のCT（CT値が下がる）

![CT of emphysema](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/ct_lung_emphysema.jpg)

終末期の肺気腫（肺胞の壁が壊れて空気が過多にたまり、肺が膨らむ病気）の横断面CTです。肺の中の黒い（空気だけの）領域が広がり、肺胞の壁が細くなっています。空気（−1000 HU）の割合が増えるほど、肺全体の $\mu$ とCT値は下がります。CT値は正常な肺（約−800 HU）から目安で約−950 HUへ下がります。

出典: Wikimedia Commons — [Emphysema CT.JPG](https://commons.wikimedia.org/wiki/File:Emphysema_CT.JPG) ／ 作者: PLoS Medicine ／ ライセンス: [CC BY 3.0](https://creativecommons.org/licenses/by/3.0/)

### 1-8. 造影CT：ヨード造影剤、低管電圧CT、デュアルエナジーCT

1-4で見たとおり、原子番号がほぼ同じ軟部組織どうしでは、CT値の差が数十HUにとどまります。造影前の肝臓は+50〜+70 HUで、脾臓や筋肉と大きくは変わりません。1-7の病気の多くも、造影剤を使わない画像では周りとの差が小さく写ります。そこで、血管から造影剤を入れて、血流の多い所と少ない所の差をつけます。

#### ヨード造影剤がCT値を上げる理由

CTの造影剤は、ヨード原子をベンゼン環に結び付けた水に溶ける化合物です。現在はほとんどが非イオン性の製剤で、1 mLに300〜370 mgのヨードを含みます。肘の静脈から自動注入器で毎秒数mLの速さで注入し、使う量は体重に合わせて決めます（肝臓のダイナミックCTでは体重1 kgあたり600 mgのヨードが一つの目安）。造影剤は腎臓から尿に排泄されます。

![Mass attenuation of iodine, calcium and water, and X-ray spectra](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/figures/fig_iodine_attenuation.png)

上の図は、ヨード・カルシウム・水の質量減弱係数（1 gあたりの減弱の大きさ）を、X線のエネルギーに対して描いたものです（縦軸は対数）。ヨードは原子番号が大きいので光電効果が強く、40〜60 keVでは同じ重さの水の40〜80倍X線を弱めます。33.17 keVのK吸収端では、減弱が不連続に約5.5倍に跳ね上がります。下の図は、20 cmの水（体に相当）を通ったあとの80 kVと120 kVのスペクトルです。80 kVのほうが、ヨードの減弱が大きい低いエネルギーの側に光子が集まっています。

ヨードによるCT値の上昇は、ヨードの濃度にほぼ比例します。このモデルで計算すると、120 kVで撮影したとき、1 mLあたり1 mgのヨードでCT値は約29 HU上がります（文献でも25〜30 HU程度とされます）。動脈相の大動脈がおよそ300 HUに写るのは、血液1 mLあたり10 mg程度のヨードが含まれていることにあたります。

出典: 自作（matplotlib、生成スクリプト `instructor/make_figures_1_1.py`。質量減弱係数は [NIST XCOM](https://physics.nist.gov/PhysRefData/XrayMassCoef/) の値）／ 作者: 本教材の作成者／ ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0)

#### 造影のタイミング（造影の相）

静脈から入れた造影剤は、心臓から動脈を通って全身の毛細血管に届き、血管の外の細胞の間（細胞外液）へも広がります。細胞の中には入らないので、このような造影剤を細胞外液性造影剤と呼びます。脳では血液脳関門があるので、正常な脳の組織には漏れ出しません。そのため、脳腫瘍や炎症で血液脳関門が壊れた所だけが造影されます。

どの臓器がいつ最も濃く写るかは、血液が届く順番で決まります。そこで、注入を始めてから決まった時間に撮影し、目的に合った相の画像を得ます。

![Time course of enhancement after intravenous contrast](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/figures/fig_contrast_phases.png)

図は、大動脈・門脈・肝実質のCT値の時間変化を描いた模式図です（数値は目安）。

- **動脈相**（注入開始から30〜40秒前後）：大動脈のCT値が最も高い時期です。血管を立体的に描くCT血管造影（CTA）や、肝細胞がんのように動脈から血流を多く受ける腫瘍が濃く写るのを捉えるのに使います
- **門脈相**（60〜80秒前後）：腸や脾臓を通った造影剤が門脈に集まる時期です。肝臓は血流の約4分の3を門脈から受けるので、肝実質はこの時期に最も濃く写ります。多くの臓器の評価や、転移の検出に使います
- **平衡相**（3分前後）：血管の中と細胞外液の造影剤の濃度が釣り合った時期です。造影剤が早く抜ける腫瘍や、線維の多い組織にゆっくり染み込む病変を見分けるのに使います

実際の撮影では、少量ずつ撮影しながら大動脈のCT値を監視し、決めた値を超えたら本撮影を始める方法（ボーラストラッキング）で、患者さんごとの血流の速さの違いを補います。

出典: 自作（matplotlib による模式図。生成スクリプト `instructor/make_figures_1_1.py`）／ 作者: 本教材の作成者／ ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0)

#### 造影剤使用後の腹部CTの例

![Normal contrast enhanced abdominal CT](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/images/ct_abdomen_contrast_enhanced.jpg)

ヨード造影剤を使ったあとの正常な腹部CTです。左が横断面（水平に切った向き）、右が冠状面（左右から身体を見た向き）です。ヨードは光電効果を増やしてCT値を上げるため、造影剤が集まる器官は周囲より明るく写ります。

造影前と造影後を並べた比較画像は、再利用できるライセンスでこのリポジトリに収められませんでした。ヨード造影剤の説明は[英語版Wikipediaの Iodinated contrast](https://en.wikipedia.org/wiki/Iodinated_contrast)で確認できます。

出典: Wikimedia Commons — [Normal contrast enhanced abdominal CT.jpg](https://commons.wikimedia.org/wiki/File:Normal_contrast_enhanced_abdominal_CT.jpg) ／ 作者: Kristie Guite, Louis Hinshaw, Fred Lee ／ ライセンス: [CC BY 3.0](https://creativecommons.org/licenses/by/3.0)

#### ヨード造影剤の安全上の注意

- **副作用**：多くは吐き気、嘔吐、じんま疹などの軽いものですが、まれに血圧の低下や呼吸困難を起こす重い反応（アナフィラキシー）があります。過去に造影剤で副作用が出た人や、ぜんそくのある人では起こりやすくなります
- **腎機能**：造影剤は腎臓から排泄されるので、腎機能が大きく低下した患者さんでは、造影後に腎機能が悪化するおそれがあります。検査の前に腎機能（eGFR）を確かめ、必要に応じて輸液などで予防します
- **飲んでいる薬**：ビグアナイド系の糖尿病薬（メトホルミンなど）を飲んでいる人では、腎機能が低下したときに乳酸アシドーシスを起こすおそれがあるので、休薬を検討します
- **甲状腺**：大量のヨードが体に入るので、甲状腺の検査や放射性ヨードによる治療の予定があるときは、時期を調整します

#### 低管電圧CT

管電圧を下げると、X線のスペクトルがヨードのK吸収端に近い低いエネルギーの側へ移り、ヨードの減弱が大きくなります。

![CT number of iodine versus tube voltage](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/figures/fig_iodine_kv.png)

1 mLあたり10 mgのヨードを含む水のCT値を、管電圧ごとに計算した図です。120 kVで291 HUのヨードが、80 kVでは453 HUと約1.6倍に写ります。これを使うと、次のことができます。

- 同じ量の造影剤で、血管や病変をより濃く写す（CT血管造影など）
- 腎機能が低下した患者さんなどで、造影剤の量を減らしても同じ濃さに写す
- 体の小さい患者さんや小児で、被ばくを抑えながら造影のコントラストを保つ

そのかわり、低い管電圧ではX線の光子の数が減り（1-2のモデルでは、同じ管電流で80 kVの光子は120 kVの約3割）、体を通り抜ける割合も下がるので、1-6で見たノイズが増えます。管電流を上げて補いますが、体の大きい患者さんではX線管の出力が足りなくなります。そのため、体格に合わせて管電圧を選び、ノイズを抑える逐次近似再構成などと組み合わせて使います。管電圧を自動で選ぶ機能をもつ装置もあります。ビームハードニングや金属によるアーチファクト（1-4）は、低い管電圧ほど強く出ます。

出典: 自作（matplotlib、生成スクリプト `instructor/make_figures_1_1.py`）／ 作者: 本教材の作成者／ ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0)

#### デュアルエナジーCT

CT値は画素ごとに1つの数なので、成分の違う物質が同じCT値になることがあります。たとえば、1 mLあたり6.9 mgのヨードを含む水と、73 mgのカルシウムを含む水は、どちらも120 kVで200 HUに写ります。これでは、造影された血管と石灰化を区別できません。

ところが、1枚目の図で見たとおり、エネルギーを変えたときの減弱の変わり方は物質によって違います。ヨードはK吸収端と強い光電効果のために、エネルギーが上がると減弱が急に下がります。カルシウムの下がり方はそれより緩やかで、水はさらに緩やかです。そこで、2種類のエネルギーのX線でそれぞれCT値を測れば、物質を見分けられます。これがデュアルエナジーCT（dual-energy CT）です。

![Dual-energy separation of iodine and calcium, and virtual monoenergetic images](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/figures/fig_dual_energy.png)

- (a) 横軸を高いエネルギー（140 kVにスズのフィルタを付けて低いエネルギーの成分を除いたもの）でのCT値、縦軸を80 kVでのCT値とした図です。濃度を変えると、ヨードとカルシウムはそれぞれ傾きの違う直線の上に並びます。120 kVでは同じ200 HUだった2つ（白抜きの印）も、80 kVではヨード311 HUとカルシウム288 HU、高いエネルギーではヨード94 HUとカルシウム122 HUと、組が違います
- (b) 2つのエネルギーの測定から計算した仮想単色X線画像（virtual monoenergetic image、VMI）のCT値です。1 mLあたり2 mgのヨードは、70 keVで52 HUですが、40 keVでは165 HUになります

一般に、画素ごとの2つのCT値から、その画素を2種類の基準物質（たとえば水とヨード）がそれぞれどれだけ含まれているかを、2つの式から2つの未知数を解いて求めます。これを物質弁別（material decomposition）と呼びます。物質弁別から、次のような画像が得られます。

- **ヨード密度画像（ヨードマップ）**：画素ごとのヨードの濃度を表した画像です。肺血栓塞栓症で血流が途絶えた肺の領域や、腫瘍の造影の程度を数値で評価できます
- **仮想非造影画像（VNC）**：ヨードの分を取り除いた、造影前に相当する画像です。造影前の撮影を省いて、被ばくを減らせることがあります
- **仮想単色X線画像**：単一のエネルギーのX線で撮ったと仮定した画像です。低いkeVでは(b)のようにヨードのコントラストが強くなり、高いkeVではビームハードニングや金属のアーチファクトが減ります
- **尿酸とカルシウムの区別**：痛風の関節にたまった尿酸の結晶や、尿路結石の成分（尿酸結石かカルシウムを含む結石か）を見分けられます

出典: 自作（matplotlib、生成スクリプト `instructor/make_figures_1_1.py`。質量減弱係数は [NIST XCOM](https://physics.nist.gov/PhysRefData/XrayMassCoef/) の値）／ 作者: 本教材の作成者／ ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0)

#### デュアルエナジーCTの実装方法

2種類のエネルギーのデータを集める方法は、大きく分けて4つあります。どれも「低いエネルギー」と「高いエネルギー」の2組の投影データを作る点は同じですが、X線管の側で分けるか、検出器の側で分けるかが違います。

![Four ways to implement dual-energy CT](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/figures/fig_dect_methods.png)

- **(a) 2管球型**（dual-source）：2組のX線管と検出器を、ガントリーの中に約90°ずらして載せます。一方を80〜100 kV、もう一方を140〜150 kVにして、高い側にはスズ（Sn）のフィルタを付けて低いエネルギーの成分を取り除きます。2つのエネルギーをほぼ同時に集められ、管電圧と管電流をそれぞれ最適に設定できます。ただし、2つ目の検出器は小さいので、デュアルエナジーの情報が得られる範囲（FOV）が中心部の直径30 cm余りに限られます。また、2つのデータは90°ずれた角度から集めるので、同じ位置のデータがそろうまでに4分の1回転分の時間差があります
- **(b) 高速管電圧切り替え型**（rapid kV switching）：1つのX線管の管電圧を、投影の角度ごとに80 kVと140 kVで1ミリ秒未満の間隔で切り替えます。検出器は1つで、全体のFOVでデュアルエナジーの情報が得られます。一方で、管電圧を切り替えても管電流やフィルタは同じ速さで切り替えられないので、2つのスペクトルの重なりが大きく、低い側のノイズが多くなりやすい方式です
- **(c) 2層検出器型**（dual-layer detector）：X線管は1つで、通常どおり120 kVで撮影します。検出器を上下2層のシンチレータにし、上の層（イットリウム系のガーネット）が低いエネルギーの光子を主に吸収し、通り抜けた高いエネルギーの光子を下の層（ガドリニウム酸硫化物）が吸収します。すべての撮影でデュアルエナジーのデータが同時に得られるので、撮影のあとで必要になったときに解析できます。そのかわり、2つの層が受け取るエネルギーの差は小さくなります
- **(d) 光子計数型**（photon-counting。1-5）：テルル化カドミウムなどの半導体で光子を1個ずつ電気のパルスに変え、パルスの高さ（光子のエネルギー）をしきい値と比べて、エネルギーの区間（ビン）ごとに数えます。しきい値を2つ以上設定すれば、1回の撮影から2つ以上のエネルギーのデータが得られます

このほか、同じ範囲を低い管電圧と高い管電圧で2回続けて撮影する方法もあります。特別な装置は要りませんが、2回の撮影のあいだに体が動くと、2組のデータがずれてしまいます。

![Detected spectra for each dual-energy method](https://raw.githubusercontent.com/Nakaura-T/Medical_Imaging_Seminar_Public/main/hiroshima/1_modalities/figures/fig_dect_spectra.png)

方式ごとに、体（水20 cm）を通ったあとに検出器が受け取るエネルギーの分布を、低い側（青）と高い側（橙）に分けて計算した図です。縦の点線は平均エネルギー、「overlap」は2つの分布が重なっている割合です。2つの分布が離れているほど、物質ごとの減弱の変わり方の違いを捉えやすく、物質を見分ける精度が上がります。

- スズのフィルタを付けた2管球型は、平均エネルギーの差が43 keVと最も大きく、重なりは13%です
- フィルタのない管電圧切り替え型は、差が27 keV、重なりが43%です
- 2層検出器型は、差が12 keV、重なりが71%と、分離は最も小さくなります
- 光子計数型は、この図では理想的に0%ですが、実際には1個の光子の電荷が隣の素子に分かれる現象（電荷共有）などで、区間どうしが一部重なります

出典: 自作（matplotlib、生成スクリプト `instructor/make_figures_1_1.py`。検出器の材料の質量減弱係数は [NIST XCOM](https://physics.nist.gov/PhysRefData/XrayMassCoef/) の値。2層検出器の上の層は Y3Al5O12 の 0.5 mm、下の層は Gd2O2S の 1.5 mm とみなした簡単なモデル）／ 作者: 本教材の作成者／ ライセンス: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0)

## 2. 動かしてみる

教員が用意したコードを、上から順に実行します。

### 2-1. 減弱の式をグラフにする

脂肪、水、骨を厚さ0〜10 cm通り抜けたあとのX線の強さを比べます。右のグラフは縦軸を対数にしたもので、減弱が指数関数であれば直線になります。

In [ ]:
# 厚さ 0〜10 cm の物質を通り抜けたあとのX線の強さ（入射したときの強さを 1 とする）
thickness = np.linspace(0, 10, 200)  # cm
mu = {"fat": 0.18, "water": 0.21, "bone": 0.57}  # 60 keV 付近の線減弱係数の目安（1/cm）

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for name, value in mu.items():
    intensity = np.exp(-value * thickness)
    axes[0].plot(thickness, intensity, label=name)
    axes[1].semilogy(thickness, intensity, label=name)
for ax in axes:
    ax.set_xlabel("Thickness (cm)")
    ax.set_ylabel("I / I0")
    ax.legend()
axes[0].set_title("Linear scale")
axes[1].set_title("Log scale")
plt.tight_layout()
plt.show()

for name, value in mu.items():
    print(f"{name}: 5 cm 通り抜けたあとの強さ = {np.exp(-value * 5):.3f}")

### 2-2. ファントム画像から投影データ（サイノグラム）を作る

0〜180度を1度ずつ、180方向から投影します。`radon` は、画像を各方向から投影したサイノグラムを返す関数です。

In [ ]:
image = shepp_logan_phantom()  # 400×400 の数値ファントム
angles = np.linspace(0.0, 180.0, 180, endpoint=False)  # 投影の角度（度）
sinogram = radon(image, theta=angles)

print("image:", image.shape, " sinogram:", sinogram.shape, "（検出器の位置の数, 角度の数）")

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].imshow(image, cmap="gray")
axes[0].set_title("Phantom")
axes[1].imshow(sinogram, cmap="gray", aspect="auto", extent=(0, 180, sinogram.shape[0], 0))
axes[1].set_title("Sinogram")
axes[1].set_xlabel("Projection angle (deg)")
axes[1].set_ylabel("Detector position")
plt.tight_layout()
plt.show()

### 2-3. 投影データから断面画像を再構成する（フィルタなし・あり）

`iradon` は逆投影で再構成する関数です。`filter_name=None` がフィルタなしの逆投影、`filter_name="ramp"` がフィルタ補正逆投影法です。

In [ ]:
recon_plain = iradon(sinogram, theta=angles, filter_name=None)
recon_fbp = iradon(sinogram, theta=angles, filter_name="ramp")

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, img, title in zip(axes, [image, recon_plain, recon_fbp],
                          ["Original", "Back projection (no filter)", "Filtered back projection"]):
    ax.imshow(img, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

# フィルタ補正逆投影法の結果と元の画像の差（平均二乗誤差の平方根）
rmse = np.sqrt(np.mean((recon_fbp - image) ** 2))
print(f"RMSE（フィルタ補正逆投影法）: {rmse:.4f}")

### 2-4. ビームハードニングを再現する

2-2と2-3では、$\mu$ がエネルギーによらないものとして投影を作っていました。実際のX線は連続スペクトルなので、そうはなりません。ここでは、低いエネルギーの成分と高いエネルギーの成分が半分ずつ混ざったX線を仮定して、一様な水の円柱を投影・再構成します。2つの成分の線減弱係数は、違いが画像で見えるように、実際より大きく離した仮の値です。

In [ ]:
# 連続スペクトルを、低いエネルギーの成分と高いエネルギーの成分の2つだけで模す
n = 256
pixel_cm = 20.0 / n  # 画像の1辺を20 cmとする
yy, xx = np.mgrid[:n, :n] - n / 2
cylinder = ((xx**2 + yy**2) < (0.4 * n) ** 2).astype(float)  # 直径16 cmの一様な円柱

mu_low, mu_high = 0.45, 0.17  # 2つの成分に対する線減弱係数（1/cm、説明のために仮定した値）
angles_bh = np.linspace(0.0, 180.0, 180, endpoint=False)
path = radon(cylinder, theta=angles_bh) * pixel_cm  # X線が円柱の中を通った長さ（cm）

# 検出器に届く強さは2つの成分の和。その対数が、装置が投影値として使う量になる
intensity = 0.5 * np.exp(-mu_low * path) + 0.5 * np.exp(-mu_high * path)
proj_poly = -np.log(intensity)
proj_mono = 0.5 * (mu_low + mu_high) * path  # 減弱がエネルギーによらない場合

# 再構成した値を pixel_cm で割って、1 cm あたりの線減弱係数に戻す
recon_poly = iradon(proj_poly, theta=angles_bh, filter_name="ramp") / pixel_cm
recon_mono = iradon(proj_mono, theta=angles_bh, filter_name="ramp") / pixel_cm

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, img, title in zip(axes[:2], [recon_mono, recon_poly],
                          ["Single energy", "Two energies (beam hardening)"]):
    im = ax.imshow(img, cmap="gray", vmin=0.20, vmax=0.34)
    ax.set_title(title)
    ax.axis("off")
    fig.colorbar(im, ax=ax, fraction=0.046)

center = n // 2
# 中心を通る5行の平均を、横方向のプロファイルとして描く
axes[2].plot(recon_mono[center - 2:center + 3].mean(axis=0), label="Single energy")
axes[2].plot(recon_poly[center - 2:center + 3].mean(axis=0), label="Two energies")
axes[2].set_xlabel("Position (pixel)")
axes[2].set_ylabel("mu (1/cm)")
axes[2].set_title("Profile through the center")
axes[2].legend()
plt.tight_layout()
plt.show()

edge = center + int(0.3 * n)  # 円柱の内側で、中心から外れた位置
print(f"単一エネルギー: 中心 {recon_mono[center, center]:.3f} /cm, 周辺 {recon_mono[center, edge]:.3f} /cm")
print(f"2つの成分:      中心 {recon_poly[center, center]:.3f} /cm, 周辺 {recon_poly[center, edge]:.3f} /cm")

## 3. AIに頼んでみる

頼む前に、ソース管理（`Ctrl+Shift+G`）でここまでの状態をコミットします。AIが変更した後は、差分を見て採用するかどうかを決めます。頼み方は `setup/03_ai_assistant.md` を参照してください。

発展課題は、基本課題を終えた人が取り組みます。

### 基本課題1：投影の方向の数を変える

投影の角度の数を 180、36、8 に減らしてフィルタ補正逆投影法で再構成し、画像を並べて表示するよう頼みます。それぞれのRMSEも表示してもらいます。

### 基本課題2：フィルタの役割を聞く

2-3で、フィルタなしの画像がぼやける理由と、フィルタによってぼやけが消える理由を、解説役（`tutor`）に質問します。答えを読んで、1-6の解説と食い違う点がないか確かめます。

### 発展課題1：金属アーチファクトを再現する

ファントム画像に、非常に値の大きい小さな領域（体内の金属を想定）を2か所加えてから投影・再構成し、画像に何が起きるか確かめます。

### 発展課題2：エネルギーによる減弱の違いを見る

1-4の関係（光電効果は $Z^3 / E^3$ に比例、コンプトン散乱はほぼ密度で決まりエネルギーとともに緩やかに減る）だけを使った簡単なモデルで、水、骨、ヨード造影剤の減弱が30〜120 keVでどう変わるかをグラフにするよう頼みます。2つの仕組みの寄与を分けて描いてもらいます。

実際の測定値とは合いませんが、どの物質でどちらの仕組みが効くかの傾向を確かめられます。実測値との違いがどこに出るかも、解説役に聞いてみてください。

## 4. AIの答えを確かめる

確認できた項目は `[ ]` を `[x]` に書き換えます。

- [ ] サイノグラムの横軸と縦軸が何を表すかを説明できる
- [ ] 投影の数を減らしたとき、画質がどう変わったかを図で確認した
- [ ] 投影の数とRMSEの関係が、図の見た目と矛盾していないことを確認した
- [ ] AIの説明（基本課題2）と1-6の解説を照らし合わせた
- [ ] 骨が白く写る理由を、密度と原子番号のどちらで説明できるか確かめた
- [ ] 2-4で、一様な円柱の中心のCT値が周辺より低くなることを、図と数値で確認した

## 5. 振り返り

| 項目 | 記入欄 |
|---|---|
| 使ったプロンプト | |
| AIの答えで直した点・採用しなかった点 | |
| この回で分かったこと | |
| まだ分からないこと | |

記入したら保存してコミットします。提出のしかたは授業で指示します。